In [17]:
import pyspark
import re
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('BigD_OurD').getOrCreate()
sc = spark.sparkContext

In [26]:
# Initialization of the whole RDD
def csvFilter(line):
    out = []
    if "\"" not in line:
        return line.split(",")
    else:
        start = 0
        indices = [match.start() for match in re.finditer(r",", line)]
        while len(indices) > 10:
            indices.pop(1)
        indices.append(-1)
        for end in indices:
             out.append(line[start:end])
             start = end+1
        return out

game_rdd = sc.textFile("vgsales.csv").map(csvFilter)
header = game_rdd.first()
game_rdd = game_rdd.filter(lambda x: x != header)

In [29]:
###################################
# Genre to Global Sale Comparison #
#    by Keiron Mirandilla         #
###################################

genre_sales = game_rdd.map(lambda x: [x[4], float(x[-1])])
summed_sales = genre_sales.reduceByKey(lambda a,b: a+b).sortBy(lambda x: x[1], ascending=False)
print(summed_sales.collect())

[('Action', 1744.8300000000072), ('Sports', 1330.5500000000097), ('Shooter', 1032.930000000001), ('Role-Playing', 926.3300000000017), ('Platform', 829.6199999999994), ('Misc', 806.2400000000013), ('Racing', 731.6500000000013), ('Fighting', 448.89999999999986), ('Simulation', 390.1), ('Puzzle', 242.8699999999999), ('Adventure', 234.77000000000012), ('Strategy', 174.1499999999998), ('"Destination Software', 0.03), ('"mixi', 0.0), ('"Interworks Unlimited', 0.0)]


In [24]:
from pyspark.sql.functions import sum as _sum, desc


platform_total_sales = df_cleaned.groupBy('Platform') \
    .agg(_sum('Global_Sales').alias('Total_Global_Sales'))

# rp
platform_total_sales = platform_total_sales.repartitionByRange(4, 'Total_Global_Sales')

# sort by most sales most to least top 10
platform_total_sales = platform_total_sales.orderBy(desc('Total_Global_Sales'))

platform_total_sales.show(10)

top_platform = platform_total_sales.first()
print(f"The platform with the most global sales is {top_platform['Platform']} with {top_platform['Total_Global_Sales']:.2f} million units.")
print(f"Number of partitions: {platform_total_sales.rdd.getNumPartitions()}")

AttributeError: 'PipelinedRDD' object has no attribute 'agg'